# FFTop GPU Kernel Sweep (Colab)

Runtime → **Change runtime type → GPU** (T4 is enough).

File → Open notebook → GitHub → `nguyen-trinhtk/FFTop` → `gpu-notebooks/stockham_sweep_colab.ipynb`.
The clone cell pulls `https://github.com/nguyen-trinhtk/FFTop.git` (`main`) into `/content/FFTop`.

Builds FFTop with CUDA and sweeps three kernel strategies:

- **cooley-tukey naive** — in-place Cooley–Tukey in global memory
- **stockham naive** — self-sorting Stockham, ping-pong global memory
- **stockham shared** — Stockham in shared memory (four-step when \(N\) exceeds the tile)

Metrics (device kernels only, no H2D/D2H):

- runtime (ms)
- effective bandwidth (GB/s) = estimated global traffic / time
- estimated global memory traffic (bytes)
- throughput (Gsamples/s) = \(N / t\)

In [ ]:
!nvidia-smi

In [ ]:
!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

In [ ]:
import os
from pathlib import Path

ROOT = Path("/content/FFTop")
REPO_URL = "https://github.com/nguyen-trinhtk/FFTop.git"
BRANCH = "main"

if ROOT.exists():
    !git -C {ROOT} fetch origin
    !git -C {ROOT} checkout {BRANCH}
    !git -C {ROOT} pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {ROOT}

os.chdir(ROOT)
print("repo:", ROOT.resolve())
!git rev-parse --short HEAD && git status -sb

In [ ]:
import subprocess

cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0]
arch = cap.replace(".", "")
print("CUDA arch:", arch)

!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DFFTOP_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={arch}
!cmake --build build --target fftop_bench fftop_tests -j

In [ ]:
!python3 -m pip -q install -r bench/requirements.txt

In [ ]:
import yaml
from pathlib import Path

# Shrink the sweep on a slow GPU. Default yaml is min_k=10, max_k=24.
MAX_K = 22  # 2**22 * 16 bytes ≈ 64 MiB per buffer

cfg_path = Path("bench/config/gpu_stockham.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg.setdefault("defaults", {})["max_k"] = MAX_K
run_cfg = Path("/tmp/gpu_stockham_colab.yaml")
run_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))

!python3 bench/bench.py --check --config {run_cfg}
!ctest --test-dir build --output-on-failure -R GPUBackend || true
!python3 bench/bench.py --bench build/bench/fftop_bench --config {run_cfg}

In [ ]:
import pandas as pd
from pathlib import Path

log_dir = sorted(Path("log").glob("*"))[-1]
csv_path = log_dir / "bench.csv"
df = pd.read_csv(csv_path, comment="#")
df["k"] = df["size"].map(lambda n: int(n).bit_length() - 1)
print("Using:", csv_path)
df.head()

In [ ]:
import matplotlib.pyplot as plt

name_map = {
    "cooley-tukey": "cooley-tukey naive",
    "stockham-global": "stockham naive",
    "stockham-shared": "stockham shared",
}
df["variant"] = df["traversal"].map(name_map).fillna(df["traversal"])

metrics = [
    ("ms", "Runtime (ms)", True),
    ("effective_bandwidth_gbps", "Effective Bandwidth (GB/s)", False),
    ("throughput_gsamples_s", "Throughput (Gsamples/s)", False),
    ("global_mem_bytes", "Estimated Global Memory Traffic (bytes)", False),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.ravel()
for ax, (col, title, invert) in zip(axes, metrics):
    for name, g in df.groupby("variant"):
        g = g.sort_values("size")
        ax.plot(g["size"], g[col], marker="o", label=name)
    ax.set_xscale("log", base=2)
    if col == "global_mem_bytes":
        ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("N")
    ax.grid(True, which="both", linestyle=":", linewidth=0.6)
    if invert:
        ax.set_ylabel("lower is better")
    else:
        ax.set_ylabel("higher is better")
axes[0].legend()
plt.show()

In [ ]:
pivot_cols = ["ms", "effective_bandwidth_gbps", "throughput_gsamples_s", "global_mem_bytes"]
show_n = [2**k for k in (12, 16, 20) if 2**k in set(df["size"])]
view = df[df["size"].isin(show_n)][["variant", "size", *pivot_cols]].sort_values(["size", "variant"])
view

In [ ]:
from IPython.display import Image, display

metrics_png = log_dir / "gpu-kernel-strategies_metrics.png"
if metrics_png.is_file():
    display(Image(filename=str(metrics_png)))
else:
    print("no combined plot at", metrics_png)